In [2]:
# Imports et chargement des données préparées
import pandas as pd

df_clean = pd.read_parquet("../data/processed/transactions_clean.parquet")
churn_labels = pd.read_parquet("../data/processed/churn_labels.parquet")

print(df_clean.shape)
print(churn_labels.shape)
df_clean.head()

(724452, 9)
(4535, 4)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [3]:
# RFM calculé uniquement sur la fenêtre d'observation (avant cutoff) -> pas de fuite de données
# Recency par rapport au cutoff, pas à la dernière date du dataset -> pas de biais de censure
cutoff = pd.Timestamp('2011-06-10')
obs = df_clean[df_clean['InvoiceDate'] < cutoff]

rfm = obs.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (cutoff - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

rfm_final = rfm.merge(churn_labels, on='Customer ID')
rfm_final = rfm_final[rfm_final['Fidele']]

print(rfm_final.shape)
rfm_final.describe()

(2349, 7)


,Customer ID,Recency,Frequency,Monetary,NbCommandesObs
count,2349.000000,2349.000000,2349.000000,2349.000000,2349.000000
mean,15538.496807,105.626224,8.912303,3960.906854,8.912303
std,1576.493220,104.117869,12.117854,12451.828472,12.117854
min,12346.000000,0.000000,3.000000,86.150000,3.000000
25%,14177.000000,21.000000,4.000000,983.100000,4.000000
50%,15573.000000,66.000000,6.000000,1794.320000,6.000000
75%,16916.000000,185.000000,10.000000,3538.700000,10.000000
max,18287.000000,541.000000,206.000000,441293.980000,206.000000


In [4]:
# Coup d'oeil sur les plus gros CA -> repérer d'éventuels comptes B2B avant de scorer
rfm_final.sort_values('Monetary', ascending=False).head(5)

,Customer ID,Recency,Frequency,Monetary,Churned,NbCommandesObs,Fidele
4381,18102,0,102,441293.98,False,102,True
712,13694,0,120,162669.85,False,120,True
3887,17511,23,44,120291.57,False,44,True
1838,15061,1,109,114563.55,False,109,True
3842,17450,9,19,108182.56,False,19,True


In [5]:
# Détection des comptes B2B/revendeurs : quantité moyenne/ligne élevée (P98) ET montant élevé (P95)
# Croiser les 2 signaux évite d'exclure un client qui achète juste beaucoup d'articles pas chers
qty_par_client = obs.groupby('Customer ID')['Quantity'].mean().reset_index()
qty_par_client.columns = ['Customer ID', 'QtyMoyenneParLigne']

qty_seuil = qty_par_client['QtyMoyenneParLigne'].quantile(0.98)
monetary_seuil = rfm_final['Monetary'].quantile(0.95)

rfm_final = rfm_final.merge(qty_par_client, on='Customer ID')
b2b_mask = (rfm_final['QtyMoyenneParLigne'] > qty_seuil) & (rfm_final['Monetary'] > monetary_seuil)

print(f"Comptes B2B détectés : {b2b_mask.sum()}")
rfm_final = rfm_final[~b2b_mask].drop(columns=['QtyMoyenneParLigne'])
print(f"Table RFM finale (retail uniquement) : {rfm_final.shape}")

Comptes B2B détectés : 20
Table RFM finale (retail uniquement) : (2329, 7)


In [6]:
# Score RFM par quartiles (R inversé : achat récent = score élevé = bon client)
# .rank(method='first') évite les erreurs de qcut quand plusieurs clients ont la même valeur
rfm_final['R_score'] = pd.qcut(rfm_final['Recency'].rank(method='first'), 4, labels=[4, 3, 2, 1]).astype(int)
rfm_final['F_score'] = pd.qcut(rfm_final['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm_final['M_score'] = pd.qcut(rfm_final['Monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)

rfm_final['RFM_Score'] = rfm_final['R_score'] + rfm_final['F_score'] + rfm_final['M_score']

rfm_final[['Customer ID','Recency','Frequency','Monetary','R_score','F_score','M_score','RFM_Score']].head(10)

,Customer ID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
1,12747,15,21,7171.93,4,4,4,12
2,12748,1,206,29415.21,4,4,4,12
3,12749,30,5,3665.58,3,2,4,9
4,12820,143,8,1917.64,2,3,3,8
5,12823,71,16,5736.50,2,4,4,10
6,12826,133,8,2178.13,2,3,3,8
7,12829,153,3,385.30,2,1,1,4
8,12831,79,3,451.11,2,1,1,4
9,12835,244,41,6043.31,1,4,4,9
10,12836,36,13,5634.16,3,4,4,11


In [7]:
# Vérifier que le score RFM sépare bien le churn (validation de l'hypothèse de faisabilité)
rfm_final.groupby('RFM_Score')['Churned'].mean().round(3) * 100

RFM_Score
3     59.9
4     55.6
5     44.1
6     39.5
7     31.4
8     22.1
9     18.6
10    13.2
11     4.6
12     2.6
Name: Churned, dtype: float64

In [ ]:
# Segments actionnables pour le CRM, basés sur le score RFM total (3-12)
rfm_final['Segment'] = pd.cut(
    rfm_final['RFM_Score'],
    bins=[2, 5, 8, 12],
    labels=['À risque critique', 'À surveiller', 'Sain']
)

print(rfm_final.groupby('Segment').agg(
    NbClients=('Customer ID', 'count'),
    TauxChurn=('Churned', 'mean'),
    CA_moyen=('Monetary', 'mean')
).round(3))

import os
os.makedirs("../../data/processed", exist_ok=True)
rfm_final.to_parquet("../../data/processed/rfm_scores.parquet", index=False)
print("Sauvegardé :", rfm_final.shape)

                   NbClients  TauxChurn  CA_moyen
Segment                                          
À risque critique        657      0.524   878.658
À surveiller             772      0.316  1775.440
Sain                     900      0.101  6530.620
Sauvegardé : (2329, 12)


C:\Users\Moi\AppData\Local\Temp\ipykernel_19204\3817253729.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(rfm_final.groupby('Segment').agg(


: 